In [ ]:
import os
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from tensorflow.keras.utils import to_categorical, load_img
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
TRAIN_DIR = "../data/FER2013/train"
TEST_DIR = "../data/FER2013/test"

print("Train directory exists:", os.path.exists(TRAIN_DIR))
print("Test directory exists:", os.path.exists(TEST_DIR))
print("Dataset folders:", os.listdir("../data/FER2013"))

In [ ]:
def createdataframe(directory):
    image_paths = []
    labels = []

    for label in os.listdir(directory):
        label_path = os.path.join(directory, label)

        if os.path.isdir(label_path):
            for imagename in os.listdir(label_path):
                image_path = os.path.join(label_path, imagename)
                image_paths.append(image_path)
                labels.append(label)

            print(label, "completed")

    return image_paths, labels

In [ ]:
# Create train dataframe
train = pd.DataFrame()
train["image"], train["label"] = createdataframe(TRAIN_DIR)

# Create test dataframe
test = pd.DataFrame()
test["image"], test["label"] = createdataframe(TEST_DIR)

print("Train dataframe:")
print(train)

print("\nTest dataframe:")
print(test)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
# Show one image from train dataframe
index = 0
img_path = train["image"][index]
label = train["label"][index]

print("Image path:", img_path)
print("Label:", label)

img = load_img(img_path, color_mode="grayscale", target_size=(48, 48))
plt.imshow(img, cmap="gray")
plt.title(label)
plt.axis("off")
plt.show()

In [ ]:
# Show multiple random images from train dataframe
sample_data = train.sample(12, random_state=42).reset_index(drop=True)

plt.figure(figsize=(12, 8))

for i in range(12):
    img_path = sample_data["image"][i]
    label = sample_data["label"][i]

    img = load_img(img_path, color_mode="grayscale", target_size=(48, 48))

    plt.subplot(3, 4, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def extract_features(images):
    features = []

    for image in tqdm(images):
        img = load_img(image, color_mode="grayscale", target_size=(48, 48))
        img = np.array(img)
        features.append(img)

    features = np.array(features)
    features = features.reshape(len(features), 48, 48, 1)

    return features


train_features = extract_features(train["image"])
test_features = extract_features(test["image"])

print("Train features shape:", train_features.shape)
print("Test features shape:", test_features.shape)

In [ ]:
x_train = train_features / 255.0
x_test = test_features / 255.0

print("x_train min/max:", x_train.min(), x_train.max())
print("x_test min/max:", x_test.min(), x_test.max())

In [ ]:
le = LabelEncoder()
le.fit(train["label"])

y_train = le.transform(train["label"])
y_test = le.transform(test["label"])

y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

emotion_labels = list(le.classes_)

print("Emotion labels:", emotion_labels)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
model = Sequential()

model.add(Input(shape=(48, 48, 1)))

model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.30))

model.add(Conv2D(256, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(Conv2D(256, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.35))

model.add(Conv2D(512, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.40))

model.add(Flatten())

model.add(Dense(512, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.50))

model.add(Dense(256, activation="relu"))
model.add(BatchNormalization())
model.add(Dropout(0.40))

model.add(Dense(7, activation="softmax"))

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
os.makedirs("../models/FER2013", exist_ok=True)

callbacks = [
    ModelCheckpoint(
        "../models/FER2013/best_fer2013_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_accuracy",
        patience=15,
        restore_best_weights=True,
        mode="max",
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
history = model.fit(
    x=x_train,
    y=y_train,
    batch_size=64,
    epochs=100,
    validation_data=(x_test, y_test),
    callbacks=callbacks
)

In [ ]:
# Training and validation accuracy graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig("../img/fer2013_training_accuracy_graph.png", dpi=300)
plt.show()

# Training and validation loss graph
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig("../img/fer2013_training_loss_graph.png", dpi=300)
plt.show()

In [ ]:
# Predict test data
y_pred_prob = model.predict(x_test)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [accuracy, precision, recall, f1]
})

print("Overall Metrics")
print(metrics_df)

metrics_df.to_csv("../results/fer2013_overall_metrics.csv", index=False)

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=emotion_labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("Classification Report")
print(report_df)

report_df.to_csv("../results/fer2013_classification_report.csv")

In [ ]:
import seaborn as sns

cm = confusion_matrix(y_true, y_pred, normalize='true')

# Save as CSV for Streamlit app
cm_df = pd.DataFrame(cm, index=emotion_labels, columns=emotion_labels)
cm_df.to_csv("../results/fer2013_confusion_matrix_normalized.csv")


# Plot with Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='.1%',
    cmap='Blues',
    xticklabels=emotion_labels,
    yticklabels=emotion_labels,
    linewidths=0.5,
    linecolor='white',
    vmin=0, 
    vmax=1,
    cbar_kws={'label': 'Percentage'}
)
plt.title('Normalized Confusion Matrix - Expression Recognition', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Save high-res image
plt.savefig("../img/fer2013_confusion_matrix_normalized.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
correct_indices = np.where(y_true == y_pred)[0]

plt.figure(figsize=(10, 8))

for i, idx in enumerate(correct_indices[:9]):
    img = x_test[idx].reshape(48, 48)

    true_label = emotion_labels[y_true[idx]]
    pred_label = emotion_labels[y_pred[idx]]

    plt.subplot(3, 3, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")

plt.suptitle("Sample Correct Predictions")
plt.tight_layout()
plt.savefig("../results/fer2013_sample_correct_predictions.png", dpi=300)
plt.show()

In [ ]:
wrong_indices = np.where(y_true != y_pred)[0]

plt.figure(figsize=(10, 8))

for i, idx in enumerate(wrong_indices[:9]):
    img = x_test[idx].reshape(48, 48)

    true_label = emotion_labels[y_true[idx]]
    pred_label = emotion_labels[y_pred[idx]]

    plt.subplot(3, 3, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")

plt.suptitle("Sample Wrong Predictions")
plt.tight_layout()
plt.savefig("../results/fer2013_sample_wrong_predictions.png", dpi=300)
plt.show()

In [ ]:
# Save final model architecture and weights
model_json = model.to_json()

with open("../models/FER2013/emotiondetector.json", "w") as json_file:
    json_file.write(model_json)

model.save("../models/FER2013/emotiondetector.h5")
model.save("../models/FER2013/final_fer2013_model.keras")

print("Models saved successfully.")
print("Best model saved at: models/FER2013/best_fer2013_model.keras")
print("Final model saved at: models/FER2013/final_fer2013_model.keras")
print("JSON + H5 model saved at: models/FER2013/emotiondetector.json and models/FER2013/emotiondetector.h5")

In [ ]:
# Load best trained model
model = load_model("../models/FER2013/best_fer2013_model.keras")

# Use same labels from LabelEncoder
emotion_labels = list(le.classes_)

print("Model loaded successfully")
print("Emotion labels:", emotion_labels)


def get_sentiment(emotion):
    """Convert predicted emotion into a simple sentiment category."""

    if emotion in ["happy", "surprise"]:
        sentiment = "Positive"
        text = "The detected facial expression suggests a positive emotional state."

    elif emotion in ["angry", "disgust", "fear", "sad"]:
        sentiment = "Negative"
        text = "The detected facial expression suggests a negative emotional state."

    elif emotion == "neutral":
        sentiment = "Neutral"
        text = "The detected facial expression suggests a neutral emotional state."

    else:
        sentiment = "Unknown"
        text = "Sentiment could not be determined."

    return sentiment, text


def preprocess_image(image_path):
    """Load one image, convert it to grayscale, resize it to 48x48, normalize it, and reshape it for CNN input."""

    img = load_img(image_path, color_mode="grayscale", target_size=(48, 48))
    img = np.array(img)
    img = img.reshape(1, 48, 48, 1)
    img = img / 255.0

    return img


def predict_single_image(image_path, original_label):
    """Predict emotion and sentiment for one image."""

    print("Original image is of:", original_label)

    img = preprocess_image(image_path)

    pred = model.predict(img)
    pred_index = np.argmax(pred)

    pred_label = emotion_labels[pred_index]
    confidence = np.max(pred) * 100

    sentiment, sentiment_text = get_sentiment(pred_label)

    print("Model prediction is:", pred_label)
    print("Confidence:", round(confidence, 2), "%")
    print("Sentiment:", sentiment)
    print("Sentiment Analysis:", sentiment_text)

    plt.imshow(img.reshape(48, 48), cmap="gray")
    plt.title(
        f"True: {original_label}\nPredicted: {pred_label} ({confidence:.2f}%)\nSentiment: {sentiment}"
    )
    plt.axis("off")
    plt.show()

In [ ]:
predict_single_image(
    "../data/FER2013/train/sad/42.jpg",
    "sad"
)

predict_single_image(
    "../data/FER2013/train/fear/2.jpg",
    "fear"
)

predict_single_image(
    "../data/FER2013/train/disgust/299.jpg",
    "disgust"
)

predict_single_image(
    "../data/FER2013/train/happy/7.jpg",
    "happy"
)

predict_single_image(
    "../data/FER2013/train/surprise/15.jpg",
    "surprise"
)